# Extract

In [2]:
from datetime import datetime, date

print(datetime.now().strftime("%Y%m%d"))

20260430


In [ ]:
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
import numpy as np
from datetime import datetime, date
today = str(datetime.now().strftime("%Y%m%d"))
import pyspark.pandas as ps
from pyspark.sql import SparkSession
from pyspark.sql import Row
import pandas as pd 
import glob
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("BGES") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()


def extract_daily_data() :
    #region Extraction des données
    #Charger les fichiers de personnel
    fichiers_personnel = glob.glob(f'BDD_BGES/**/PERSONNEL_{today}.txt', recursive=True)
    psdf_personnel_list = [ps.read_csv(f, sep=';',index_col='ID_PERSONNEL') for f in fichiers_personnel]
    psdf_personnel = ps.concat(psdf_personnel_list, ignore_index=False)

    #Charger les fichiers de matériel informatique
    fichiers_mat_inf = glob.glob(f'BDD_BGES/**/MATERIEL_INFORMATIQUE_{today}.txt', recursive=True)
    psdf_mat_inf_list = [ps.read_csv(f, sep=';') for f in fichiers_mat_inf]
    psdf_mat_inf_temp = ps.concat(psdf_mat_inf_list, ignore_index=True)

    #Charge les données d'impact carbone du matériel informatique
    psdf_impact_matinf = ps.read_csv('BDD_BGES/materiel_informatique_impact.csv',sep=',')
    psdf_impact_matinf = psdf_impact_matinf.rename(columns={
        'Type': 'TYPE',
        'Impact': 'IMPACT',
        'Modèle': 'MODELE'
    })

    #Fusionne les données de matériel informatique avec les données d'impact carbone
    psdf_mat_inf = psdf_mat_inf_temp.merge(
        psdf_impact_matinf,
        on=['TYPE','MODELE'],
        how='inner'
    )
    #Impact est en kg/CO2, il faut le convertir en tonnes de CO2
    psdf_mat_inf['IMPACT'] = psdf_mat_inf['IMPACT'] / 1000
    psdf_mat_inf = psdf_mat_inf.set_index('ID_MATERIELINFO')

    #Charger les fichiers de mission
    fichiers_mission = glob.glob(f'BDD_BGES/**/MISSION_{today}.txt', recursive=True)
    psdf_mission_list = [ps.read_csv(f, sep=';',index_col='ID_MISSION') for f in fichiers_mission]
    psdf_mission = ps.concat(psdf_mission_list, ignore_index=False)
    #endregion
    return psdf_personnel, psdf_mat_inf, psdf_mission
psdf_personnel, psdf_mat_inf, psdf_mission = extract_daily_data()

# Transform

In [ ]:
def transform_daily_data() :
    #region Transformation des données

    # Transformation en spark dataframe
    sdf_personnel = psdf_personnel.to_spark(index_col='ID_PERSONNEL')
    sdf_mat_inf = psdf_mat_inf.to_spark(index_col='ID_MATERIELINFO')
    sdf_mission = psdf_mission.to_spark(index_col='ID_MISSION')

    #Formatage des données dans le format des tables de la base de données

    
    FAITS_MISSION = sdf_mission.join(sdf_personnel, on='ID_PERSONNEL', how='inner').select('ID_MISSION','ID_PERSONNEL',col('VILLE').alias('SITE'),year(to_date(col('DATE_MISSION'))).alias('ANNEE'), month(to_date(col('DATE_MISSION'))).alias('MOIS'), dayofmonth(to_date(col('DATE_MISSION'))).alias('JOUR'))
    
    FAITS_MATERIEL_INFORMATIQUE = sdf_mat_inf.join(sdf_personnel, on='ID_PERSONNEL', how='inner').select('ID_PERSONNEL', 'ID_MATERIELINFO', year(to_date(col('DATE_ACHAT'))).alias('ANNEE'), month(to_date(col('DATE_ACHAT'))).alias('MOIS'), dayofmonth(to_date(col('DATE_ACHAT'))).alias('JOUR'), col('VILLE').alias('SITE'))

    SITE = sdf_personnel.select(col('VILLE').alias('SITE')).distinct()

    DATE = sdf_mission.select(col('DATE_MISSION').alias('DATE')).join(sdf_mat_inf.select(col('DATE_ACHAT').alias('DATE')), on='DATE', how='outer')
    DATE = DATE.select(to_date(col('DATE'), 'yyyy-MM-dd').alias('DATE'))
    DATE = DATE.select(year(col('DATE')).alias('ANNEE'), month(col('DATE')).alias('MOIS'), dayofmonth(col('DATE')).alias('JOUR'))

    PERSONNEL = sdf_personnel.select('ID_PERSONNEL','PAYS','FONCTION_PERSONNEL',col('DT_NAISS').alias('DATE_NAISSANCE'))

    MATERIEL_INFORMATIQUE = sdf_mat_inf.select('ID_MATERIELINFO', 'TYPE', 'MODELE', 'IMPACT')
    MATERIEL_INFORMATIQUE = MATERIEL_INFORMATIQUE.withColumn(
        "ECRAN",
        when(lower(col("TYPE")).contains("sans ecran"), "non")
        .otherwise("oui")
    ).withColumn(
        "TYPE_CLEAN",
        trim(split(lower(col("TYPE")), "sans ecran").getItem(0))
    ).select('ID_MATERIELINFO', col('TYPE_CLEAN').alias('TYPE'), 'MODELE', 'IMPACT', 'ECRAN')
    #endregion